# Capstone: The Analysis Agent

## Module 7 — Capstone Project (Notebook 3 of 6)

The Analysis Agent is Castalia Scholar's **critical thinker**. It takes the documents retrieved by the Retrieval Agent and produces structured analysis: synthesized findings, contradiction detection, evidence weighing, and gap identification.

### What You'll Build

1. **AnalysisAgent** — the complete analysis pipeline
2. **Synthesis** — extract key findings from multiple documents, group by theme
3. **Contradiction detection** — compare claims across sources, flag disagreements
4. **Evidence weighing** — rate source quality (peer-reviewed > blog > opinion)
5. **Gap identification** — what aspects of the question aren't covered?
6. **Integration with retrieval** — request more retrieval when gaps are found
7. **Tests with contradictory sources** — verify the analyzer catches conflicts

This notebook uses **reflection** (nb 07) and **critique** patterns to build a rigorous analysis engine.

---

> **Prerequisites**: Notebooks 32–33 (architecture, retrieval agent).
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~45–60 minutes.

## Part 0: Environment Setup

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. The Analyzer's Job

The Analyzer sits between retrieval and writing in the pipeline:

```
Retrieved Documents ──► ANALYZER ──► Structured Analysis
  (raw content)          │              (findings, contradictions,
                         │               gaps, evidence quality)
                         │
                    ┌────┴────┐
                    │ Methods  │
                    ├──────────┤
                    │synthesize│ ──► Key findings grouped by theme
                    │detect    │ ──► Contradictions between sources
                    │weigh     │ ──► Source quality ratings
                    │find gaps │ ──► Uncovered aspects
                    └──────────┘
```

### Why Analysis Matters

Without an analysis step:
- The Writer would just summarize documents sequentially (no synthesis)
- Contradictions between sources would go undetected
- Missing information wouldn't be flagged
- All sources would be treated equally regardless of quality

The Analyzer transforms a **pile of documents** into **structured knowledge** that the Writer can turn into a coherent report.

## 2. Data Structures for Analysis

We define the data types that flow through the analysis pipeline.

In [ ]:
# ─── Analysis Data Structures ───

@dataclass
class RetrievedDoc:
    """Represents a document from the Retrieval Agent."""
    doc_id: str
    title: str
    content: str
    source: str
    relevance_score: float = 0.0

@dataclass
class Finding:
    """A key finding extracted from analysis."""
    theme: str
    claim: str
    supporting_docs: List[str]  # doc_ids
    confidence: float = 0.0     # 0.0 to 1.0
    evidence_excerpt: str = ""

@dataclass
class Contradiction:
    """A contradiction detected between two sources."""
    claim_a: str
    source_a: str
    claim_b: str
    source_b: str
    description: str = ""
    severity: str = "moderate"  # "minor", "moderate", "major"

@dataclass
class AnalysisResult:
    """Complete analysis output — the Analyzer's product."""
    findings: List[Finding] = field(default_factory=list)
    contradictions: List[Contradiction] = field(default_factory=list)
    gaps: List[str] = field(default_factory=list)
    evidence_quality: Dict[str, float] = field(default_factory=dict)
    themes: List[str] = field(default_factory=list)

    def summary(self) -> str:
        lines = [
            f"Findings: {len(self.findings)} across {len(self.themes)} themes",
            f"Contradictions: {len(self.contradictions)}",
            f"Gaps: {len(self.gaps)}",
            f"Sources rated: {len(self.evidence_quality)}",
        ]
        return "\n".join(lines)

print("✓ Analysis data structures defined")
print("  RetrievedDoc, Finding, Contradiction, AnalysisResult")

## 3. Test Corpus — Documents with Known Properties

We create a test corpus with deliberate contradictions and known themes so we can verify the Analyzer's behavior.

In [ ]:
# ─── Test corpus: documents about AI energy efficiency ───
# Includes deliberate contradictions and known gaps for testing

test_documents = [
    RetrievedDoc(
        doc_id="t_001",
        title="Model Pruning Reduces Energy by 80%",
        content="Neural network pruning removes redundant parameters, creating smaller and faster models. "
                "Structured pruning removes entire neurons or attention heads, achieving 80-90% sparsity "
                "with minimal accuracy loss. The lottery ticket hypothesis shows that dense networks contain "
                "sparse subnetworks matching full model performance. Pruned models can run on commodity hardware, "
                "significantly reducing energy consumption during inference.",
        source="Frankle & Carlin, ICLR 2019",
        relevance_score=5
    ),
    RetrievedDoc(
        doc_id="t_002",
        title="Quantization: 4-bit Models Match Full Precision",
        content="Quantization reduces numerical precision from 32-bit to 8-bit or 4-bit integers. "
                "Post-training quantization requires no retraining. Recent 4-bit methods (GPTQ, AWQ, NF4) "
                "allow 70B parameter models on a single consumer GPU with minimal quality degradation. "
                "This reduces energy consumption by 4-8x during inference compared to full precision.",
        source="Dettmers et al., NeurIPS 2023",
        relevance_score=5
    ),
    RetrievedDoc(
        doc_id="t_003",
        title="Green AI: Efficiency Should Be Primary Metric",
        content="The Green AI movement advocates measuring computational cost alongside accuracy. "
                "Green AI distinguishes 'Red AI' (state-of-the-art through scale) from 'Green AI' "
                "(good results efficiently). Efficient architectures, progressive training, and "
                "hardware-software co-design are central. Research should report FLOPs, energy, and carbon.",
        source="Schwartz et al., CACM 2020",
        relevance_score=4
    ),
    RetrievedDoc(
        doc_id="t_004",
        title="Large Models Are Actually More Efficient Per Task",
        content="Contrary to the assumption that smaller models are more efficient, large foundation models "
                "amortize their training cost across hundreds of downstream tasks. A single GPT-4 training run "
                "replaces thousands of task-specific model training runs. When measured per-task, large models "
                "are MORE energy efficient than training separate small models for each task. The total carbon "
                "footprint of the foundation model paradigm may be LOWER than the alternative.",
        source="Bommasani et al., Stanford HAI 2021",
        relevance_score=4
    ),
    RetrievedDoc(
        doc_id="t_005",
        title="Training GPT-3 Consumed 1,287 MWh",
        content="Training GPT-3 consumed an estimated 1,287 MWh of energy — equivalent to the annual "
                "consumption of 120 US homes. The carbon footprint exceeded 300 tons of CO2. Inference costs "
                "are even more significant when models are widely deployed — total inference energy exceeds "
                "training by orders of magnitude. Large models are fundamentally energy-intensive.",
        source="Patterson et al., 2021",
        relevance_score=5
    ),
    RetrievedDoc(
        doc_id="t_006",
        title="Mixture of Experts: Scale Without Proportional Cost",
        content="MoE models activate only a subset of parameters per input. A gating network routes tokens "
                "to relevant experts. Mixtral 8x7B matches GPT-3.5 using only 13B active parameters per "
                "forward pass. MoE achieves better quality-per-FLOP than dense models, offering a path to "
                "efficient scaling.",
        source="Fedus et al., JMLR 2022",
        relevance_score=4
    ),
    RetrievedDoc(
        doc_id="t_007",
        title="Speculative Decoding Speeds Inference 2-3x",
        content="Speculative decoding uses a small draft model to generate candidate tokens, verified in "
                "parallel by the larger target model. Since most tokens are predictable, the draft model "
                "gets them right and only surprising tokens need the expensive model. This achieves 2-3x "
                "speedup without changing output quality, directly reducing inference energy costs.",
        source="Leviathan et al., ICML 2023",
        relevance_score=4
    ),
    RetrievedDoc(
        doc_id="t_008",
        title="Custom Hardware Accelerators Improve Efficiency 10x",
        content="TPUv4 pods achieve 1.1 exaflops while improving energy efficiency 2-3x over GPUs. "
                "Neuromorphic chips like Intel Loihi mimic brain spiking networks and can be orders of "
                "magnitude more efficient for certain tasks. Hardware-software co-design is essential — "
                "custom silicon designed for specific model architectures achieves 10x efficiency gains.",
        source="Jouppi et al., ISCA 2023",
        relevance_score=5
    ),
    RetrievedDoc(
        doc_id="t_009",
        title="Knowledge Distillation Compresses Models 60%",
        content="Knowledge distillation trains a smaller student model to mimic a larger teacher. "
                "The student learns from soft probability distributions, gaining richer information than "
                "hard labels provide. Distillation has compressed BERT by 40-60% while retaining 95%+ "
                "performance. Combined with quantization, distilled models are highly energy-efficient.",
        source="Hinton et al., NIPS Workshop 2015",
        relevance_score=4
    ),
    RetrievedDoc(
        doc_id="t_010",
        title="Edge AI Enables On-Device Inference",
        content="Edge AI runs models on devices rather than in the cloud, reducing latency and data center "
                "energy use. Mobile architectures like MobileNet achieve high accuracy within tight resource "
                "constraints. Recent on-device LLMs (Gemma 2B, Phi-3 Mini) bring conversational AI to phones. "
                "Quantization to 4 bits and structured pruning make this practical.",
        source="Howard et al., CVPR 2017",
        relevance_score=3
    ),
]

print(f"✓ Test corpus created: {len(test_documents)} documents")
print(f"\nKnown properties for testing:")
print(f"  - Contradiction: doc t_004 ('large models more efficient per task')")
print(f"    vs doc t_005 ('large models fundamentally energy-intensive')")
print(f"  - Themes: pruning, quantization, MoE, hardware, distillation, edge AI, green AI")
print(f"  - Known gap: no documents about data center cooling or renewable energy for AI")

## 4. Synthesis — Extracting Key Findings

The synthesis step extracts key claims from documents and groups them by theme. This transforms a list of documents into structured knowledge.

In [ ]:
# ─── AnalysisAgent: Synthesis ───

class AnalysisAgent:
    """
    Analyzes retrieved documents: synthesizes findings, detects contradictions,
    weighs evidence, and identifies gaps.
    """

    def __init__(self, generate_fn):
        self.generate_fn = generate_fn

    def synthesize(self, docs: List[RetrievedDoc], question: str) -> Tuple[List[Finding], List[str]]:
        """Extract key findings from documents, grouped by theme."""
        doc_summaries = ""
        for doc in docs:
            doc_summaries += f"\n[{doc.doc_id}] {doc.title} (Source: {doc.source})\n"
            doc_summaries += f"  {doc.content[:300]}\n"

        prompt = f"""Analyze these research documents and extract key findings grouped by theme.

Research question: {question}

Documents:
{doc_summaries}

Extract 5-8 key findings. For each finding, identify:
1. The theme it belongs to
2. The specific claim
3. Which document(s) support it (by doc_id)
4. Your confidence (0.0-1.0)

Respond as JSON:
{{
  "themes": ["theme1", "theme2", ...],
  "findings": [
    {{"theme": "...", "claim": "...", "supporting_docs": ["t_001", "t_002"], "confidence": 0.9}},
    ...
  ]
}}"""

        messages = [{"role": "user", "content": prompt}]
        response = self.generate_fn(messages, max_new_tokens=800, temperature=0.3)

        try:
            result = json.loads(response)
        except json.JSONDecodeError:
            match = re.search(r'\{.*\}', response, re.DOTALL)
            if match:
                try:
                    result = json.loads(match.group())
                except json.JSONDecodeError:
                    result = {"themes": [], "findings": []}
            else:
                result = {"themes": [], "findings": []}

        themes = result.get("themes", [])
        findings = []
        for f in result.get("findings", []):
            findings.append(Finding(
                theme=f.get("theme", "unknown"),
                claim=f.get("claim", ""),
                supporting_docs=f.get("supporting_docs", []),
                confidence=f.get("confidence", 0.5)
            ))

        return findings, themes


# Test synthesis
agent = AnalysisAgent(generate_fn=generate)

print("Running synthesis on test corpus...")
print("="*70)
findings, themes = agent.synthesize(test_documents, "What are the main approaches to making AI systems more energy-efficient?")

print(f"\nThemes identified ({len(themes)}):")
for t in themes:
    print(f"  • {t}")

print(f"\nFindings ({len(findings)}):")
for i, f in enumerate(findings, 1):
    print(f"  {i}. [{f.theme}] {f.claim}")
    print(f"     Supported by: {f.supporting_docs} | Confidence: {f.confidence}")

## 5. Contradiction Detection

The Analyzer compares claims across sources to identify disagreements. This is crucial for research quality — a report that ignores contradictions is misleading.

In [ ]:
# ─── Contradiction Detection ───

def detect_contradictions(self, docs: List[RetrievedDoc]) -> List[Contradiction]:
    """Compare claims across documents to find contradictions."""
    doc_claims = ""
    for doc in docs:
        doc_claims += f"\n[{doc.doc_id}] {doc.title}:\n  {doc.content[:250]}\n"

    prompt = f"""Analyze these documents and identify any contradictions — cases where
two sources make conflicting claims about the same topic.

Documents:
{doc_claims}

For each contradiction, identify:
1. The first claim and its source
2. The second claim and its source
3. A description of why they conflict
4. Severity: "minor" (nuance difference), "moderate" (different conclusions), "major" (directly opposing)

Respond as JSON:
{{
  "contradictions": [
    {{
      "claim_a": "...", "source_a": "doc_id",
      "claim_b": "...", "source_b": "doc_id",
      "description": "...", "severity": "moderate"
    }}
  ]
}}

If no contradictions exist, return {{"contradictions": []}}."""

    messages = [{"role": "user", "content": prompt}]
    response = self.generate_fn(messages, max_new_tokens=600, temperature=0.3)

    try:
        result = json.loads(response)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            try:
                result = json.loads(match.group())
            except json.JSONDecodeError:
                result = {"contradictions": []}
        else:
            result = {"contradictions": []}

    contradictions = []
    for c in result.get("contradictions", []):
        contradictions.append(Contradiction(
            claim_a=c.get("claim_a", ""),
            source_a=c.get("source_a", ""),
            claim_b=c.get("claim_b", ""),
            source_b=c.get("source_b", ""),
            description=c.get("description", ""),
            severity=c.get("severity", "moderate")
        ))
    return contradictions

# Bind method to class
AnalysisAgent.detect_contradictions = detect_contradictions

# Test contradiction detection
print("Running contradiction detection...")
print("="*70)
contradictions = agent.detect_contradictions(test_documents)

print(f"\nContradictions detected: {len(contradictions)}")
for i, c in enumerate(contradictions, 1):
    print(f"\n  Contradiction {i} (severity: {c.severity}):")
    print(f"    Claim A [{c.source_a}]: {c.claim_a}")
    print(f"    Claim B [{c.source_b}]: {c.claim_b}")
    print(f"    Why: {c.description}")

# Check if we caught the planted contradiction
planted = any(
    ("t_004" in c.source_a or "t_004" in c.source_b) and
    ("t_005" in c.source_a or "t_005" in c.source_b)
    for c in contradictions
)
print(f"\n{'✓' if planted else '✗'} Planted contradiction (t_004 vs t_005) {'detected' if planted else 'missed'}")

## 6. Evidence Weighing

Not all sources are equal. Peer-reviewed papers carry more weight than blog posts. The Analyzer rates source quality to help the Writer prioritize information.

In [ ]:
# ─── Evidence Weighing ───

def weigh_evidence(self, docs: List[RetrievedDoc]) -> Dict[str, float]:
    """Rate source quality for each document."""
    source_list = ""
    for doc in docs:
        source_list += f"  [{doc.doc_id}] Title: {doc.title} | Source: {doc.source}\n"

    prompt = f"""Rate the quality of each research source on a scale of 1-5.

Consider:
- Peer-reviewed conference/journal paper = 4-5
- Technical report from reputable institution = 3-4
- Industry blog or whitepaper = 2-3
- Opinion piece or unverified source = 1-2

Sources:
{source_list}

Respond as JSON: {{"ratings": {{"doc_id": score, ...}}}}"""

    messages = [{"role": "user", "content": prompt}]
    response = self.generate_fn(messages, max_new_tokens=300, temperature=0.2)

    try:
        result = json.loads(response)
        return {k: float(v) for k, v in result.get("ratings", {}).items()}
    except (json.JSONDecodeError, ValueError):
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            try:
                result = json.loads(match.group())
                return {k: float(v) for k, v in result.get("ratings", {}).items()}
            except (json.JSONDecodeError, ValueError):
                pass
        return {doc.doc_id: 3.0 for doc in docs}

AnalysisAgent.weigh_evidence = weigh_evidence

# Test evidence weighing
print("Rating source quality...")
print("="*70)
quality_ratings = agent.weigh_evidence(test_documents)

print(f"\nEvidence Quality Ratings:")
for doc in test_documents:
    rating = quality_ratings.get(doc.doc_id, 0)
    stars = "★" * int(rating) + "☆" * (5 - int(rating))
    print(f"  [{doc.doc_id}] {stars} ({rating:.0f}/5) {doc.title}")
    print(f"         Source: {doc.source}")

## 7. Gap Identification

The Analyzer identifies aspects of the research question that aren't covered by the retrieved documents. This information can trigger additional retrieval.

In [ ]:
# ─── Gap Identification ───

def identify_gaps(self, docs: List[RetrievedDoc], question: str) -> List[str]:
    """Identify aspects of the research question not covered by the documents."""
    covered_topics = ""
    for doc in docs:
        covered_topics += f"  - {doc.title}: {doc.content[:150]}...\n"

    prompt = f"""Given this research question and the documents retrieved so far, identify
what important aspects of the question are NOT covered.

Research question: {question}

Topics covered by retrieved documents:
{covered_topics}

List 2-5 specific gaps — aspects of the question that the current documents
don't address but should be included in a comprehensive answer.

Respond as JSON: {{"gaps": ["gap1", "gap2", ...]}}"""

    messages = [{"role": "user", "content": prompt}]
    response = self.generate_fn(messages, max_new_tokens=300, temperature=0.3)

    try:
        result = json.loads(response)
        return result.get("gaps", [])
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            try:
                result = json.loads(match.group())
                return result.get("gaps", [])
            except json.JSONDecodeError:
                pass
        return []

AnalysisAgent.identify_gaps = identify_gaps

# Test gap identification
print("Identifying coverage gaps...")
print("="*70)
gaps = agent.identify_gaps(test_documents, "What are the main approaches to making AI systems more energy-efficient?")

print(f"\nGaps identified ({len(gaps)}):")
for i, gap in enumerate(gaps, 1):
    print(f"  {i}. {gap}")

print(f"\nExpected gaps might include: data center cooling, renewable energy,")
print(f"  carbon offsetting, efficient data pipelines, training schedule optimization")

## 8. Full Analysis Pipeline

Now we combine all analysis steps into a single `analyze()` method that produces a complete `AnalysisResult`.

In [ ]:
# ─── Full Analysis Pipeline ───

def analyze(self, docs: List[RetrievedDoc], question: str) -> AnalysisResult:
    """Run the complete analysis pipeline."""
    print(f"  Analyzing {len(docs)} documents for: {question[:60]}...")

    # Step 1: Synthesize findings
    print("  [1/4] Synthesizing findings...")
    findings, themes = self.synthesize(docs, question)
    print(f"    → {len(findings)} findings across {len(themes)} themes")

    # Step 2: Detect contradictions
    print("  [2/4] Detecting contradictions...")
    contradictions = self.detect_contradictions(docs)
    print(f"    → {len(contradictions)} contradictions found")

    # Step 3: Weigh evidence
    print("  [3/4] Weighing evidence quality...")
    evidence_quality = self.weigh_evidence(docs)
    avg_quality = sum(evidence_quality.values()) / len(evidence_quality) if evidence_quality else 0
    print(f"    → Average source quality: {avg_quality:.1f}/5")

    # Step 4: Identify gaps
    print("  [4/4] Identifying coverage gaps...")
    gaps = self.identify_gaps(docs, question)
    print(f"    → {len(gaps)} gaps identified")

    result = AnalysisResult(
        findings=findings,
        contradictions=contradictions,
        gaps=gaps,
        evidence_quality=evidence_quality,
        themes=themes
    )
    return result

AnalysisAgent.analyze = analyze

# Run full analysis
print("Running full analysis pipeline...")
print("="*70)
analysis_result = agent.analyze(
    test_documents,
    "What are the main approaches to making AI systems more energy-efficient?"
)

print(f"\n{'='*70}")
print("ANALYSIS RESULT SUMMARY")
print(f"{'='*70}")
print(analysis_result.summary())

## 9. Stress Test — Deliberately Contradictory Sources

Let's test the Analyzer with a more extreme set of contradictions to verify it catches direct conflicts.

In [ ]:
# ─── Contradictory Source Test ───

contradictory_docs = [
    RetrievedDoc(
        doc_id="c_001",
        title="Quantization Causes Significant Quality Loss",
        content="Our comprehensive evaluation shows that 4-bit quantization consistently degrades model "
                "quality by 15-25% on complex reasoning tasks. While simpler tasks show minimal impact, "
                "the quality-efficiency tradeoff is substantial for state-of-the-art benchmarks. "
                "Researchers should be cautious about claims of 'lossless' quantization.",
        source="Zhang et al., Quantization Quality Analysis, EMNLP 2023",
        relevance_score=4
    ),
    RetrievedDoc(
        doc_id="c_002",
        title="4-bit Quantization is Nearly Lossless",
        content="Modern 4-bit quantization techniques (GPTQ, AWQ) achieve less than 1% accuracy "
                "degradation across standard benchmarks. When combined with careful calibration, "
                "quantized models are practically indistinguishable from full-precision counterparts. "
                "4-bit inference is the new standard for production deployment.",
        source="Dettmers et al., NeurIPS 2023",
        relevance_score=5
    ),
    RetrievedDoc(
        doc_id="c_003",
        title="Smaller Models Outperform Larger Ones When Well-Trained",
        content="The Chinchilla scaling laws demonstrate that many large models are undertrained. "
                "A smaller model trained on more data consistently outperforms a larger, undertrained model. "
                "Llama 13B matches GPT-3 (175B) when properly trained, proving that model size is "
                "less important than training data quality and quantity.",
        source="Hoffmann et al., NeurIPS 2022",
        relevance_score=4
    ),
    RetrievedDoc(
        doc_id="c_004",
        title="Scale Is All You Need",
        content="Emergent abilities only appear in sufficiently large models. Chain-of-thought reasoning, "
                "in-context learning, and instruction following require scale thresholds that smaller models "
                "cannot reach regardless of training data. For frontier AI capabilities, there is no "
                "substitute for model size. Scaling remains the most reliable path to capability improvement.",
        source="Wei et al., TMLR 2022",
        relevance_score=4
    ),
]

print("Running contradiction detection on deliberately conflicting sources...")
print("="*70)
contradictions_test = agent.detect_contradictions(contradictory_docs)

print(f"\nContradictions found: {len(contradictions_test)}")
for i, c in enumerate(contradictions_test, 1):
    print(f"\n  {i}. [{c.severity.upper()}] {c.description}")
    print(f"     Source A [{c.source_a}]: {c.claim_a[:80]}...")
    print(f"     Source B [{c.source_b}]: {c.claim_b[:80]}...")

expected_pairs = [("c_001", "c_002"), ("c_003", "c_004")]
for pair in expected_pairs:
    found = any(
        (pair[0] in c.source_a and pair[1] in c.source_b) or
        (pair[1] in c.source_a and pair[0] in c.source_b)
        for c in contradictions_test
    )
    print(f"\n  {'✓' if found else '✗'} Expected contradiction {pair[0]} vs {pair[1]}: {'found' if found else 'missed'}")

## 10. Integration — Analyzer Requests More Retrieval

When the Analyzer identifies gaps, it can generate additional search queries for the Retrieval Agent. This creates a feedback loop:

```
Retriever ──► Analyzer ──► Gaps found? ──YES──► New queries ──► Retriever
                              │
                              NO
                              │
                              ▼
                          Proceed to Writer
```

In [ ]:
# ─── Gap-Driven Retrieval Requests ───

def generate_gap_queries(self, gaps: List[str]) -> List[str]:
    """Convert identified gaps into search queries for additional retrieval."""
    if not gaps:
        return []

    gap_list = "\n".join(f"  {i+1}. {gap}" for i, gap in enumerate(gaps))

    prompt = f"""Convert these research gaps into search queries that could find relevant documents.

Gaps:
{gap_list}

For each gap, write one focused search query.
Respond as JSON: {{"queries": ["query1", "query2", ...]}}"""

    messages = [{"role": "user", "content": prompt}]
    response = self.generate_fn(messages, max_new_tokens=200, temperature=0.3)

    try:
        result = json.loads(response)
        return result.get("queries", [])
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            try:
                result = json.loads(match.group())
                return result.get("queries", [])
            except json.JSONDecodeError:
                pass
        return gaps  # fallback: use gap descriptions as queries

AnalysisAgent.generate_gap_queries = generate_gap_queries

# Test gap-to-query conversion
print("Converting gaps to search queries...")
print("="*70)
gap_queries = agent.generate_gap_queries(gaps)

print(f"\nOriginal gaps → Search queries:")
for i, (gap, query) in enumerate(zip(gaps, gap_queries if gap_queries else ["(no query)"]*len(gaps)), 1):
    print(f"  {i}. Gap: {gap}")
    print(f"     Query: {query}")
    print()

## 11. The Complete Process Method

This integrates all analysis steps into the `process()` method that the Orchestrator calls.

In [ ]:
# ─── Complete Process Method ───

def process(self, research_question: str, documents: List[RetrievedDoc],
            request_more_retrieval: bool = True) -> Dict[str, Any]:
    """
    Run the complete analysis pipeline. Returns structured results
    that the Writer agent will use to produce the report.
    """
    print(f"\n{'='*70}")
    print(f"  ANALYSIS AGENT — Processing")
    print(f"{'='*70}")
    print(f"  Question: {research_question[:70]}...")
    print(f"  Documents: {len(documents)}")
    print()

    # Run full analysis
    analysis = self.analyze(documents, research_question)

    # Generate additional queries if gaps found
    additional_queries = []
    if request_more_retrieval and analysis.gaps:
        print(f"\n  Generating queries for {len(analysis.gaps)} gaps...")
        additional_queries = self.generate_gap_queries(analysis.gaps)

    result = {
        "analysis": analysis,
        "additional_queries": additional_queries,
        "needs_more_retrieval": len(additional_queries) > 0
    }

    print(f"\n  ✓ Analysis complete")
    print(f"    {len(analysis.findings)} findings, {len(analysis.contradictions)} contradictions")
    print(f"    {len(analysis.gaps)} gaps, {len(additional_queries)} follow-up queries")

    return result

AnalysisAgent.process = process

# Run the complete process
print("Running complete analysis process...")
result = agent.process(
    "What are the main approaches to making AI systems more energy-efficient?",
    test_documents
)

print(f"\nNeeds more retrieval: {result['needs_more_retrieval']}")
if result['additional_queries']:
    print(f"Additional queries:")
    for q in result['additional_queries']:
        print(f"  → {q}")

## Key Takeaways

1. **Analysis transforms documents into knowledge** — raw retrieval results become structured findings, contradictions, and gaps through LLM-powered analysis.
2. **Contradiction detection is essential** — research quality requires identifying when sources disagree, not just summarizing them.
3. **Evidence weighing adds credibility** — not all sources are equal; rating source quality helps the Writer prioritize authoritative information.
4. **Gap identification enables feedback loops** — when the Analyzer finds missing coverage, it generates new queries for the Retriever.
5. **Structured output (AnalysisResult)** — typed dataclasses make the Analyzer's output predictable and type-safe for the Writer.
6. **Reflection patterns in action** — the Analyzer uses the same self-critique and evaluation patterns from notebook 07.

> **Next notebook**: We build the **Writing Agent** — structured report generation with citations and coherence.